# Deep Learning with PyTorch
## Neural Networks · CNNs · Training Best Practices · Transformers

---

## Table of Contents

1. PyTorch Fundamentals — Tensors and Autograd
2. Building Neural Networks — nn.Module
3. Training Loop — Loss, Optimizer, Backprop
4. Regularization — Dropout, Batch Norm, Weight Decay
5. Optimizers and Learning Rate Scheduling
6. Convolutional Neural Networks (CNN)
7. Transfer Learning
8. Recurrent Networks — LSTM and GRU
9. Attention and Transformer Basics
10. Practical Tips and Production Patterns

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset, Dataset
    from torch.optim import Adam, SGD, AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR, StepLR
    print(f'PyTorch version: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {DEVICE}')
    TORCH_AVAILABLE = True
except ImportError:
    print('PyTorch not installed. Run: pip install torch torchvision')
    TORCH_AVAILABLE = False

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)

if TORCH_AVAILABLE:
    torch.manual_seed(42)

# Section 1 — PyTorch Fundamentals: Tensors and Autograd

## Concept

PyTorch's core data structure is the **Tensor** — an n-dimensional array backed by GPU-accelerated computation.

**Autograd** automatically computes gradients via a dynamic computational graph.
Every operation on a `requires_grad=True` tensor is tracked, and calling `.backward()` propagates gradients back through the graph.

## Technical Deep Dive

| Concept | Description |
|---|---|
| `torch.Tensor` | N-dimensional array |
| `requires_grad=True` | Enable gradient tracking |
| `.backward()` | Compute all gradients (backprop) |
| `.grad` | Accumulated gradient (after backward) |
| `.detach()` | Remove from computation graph |
| `torch.no_grad()` | Context manager — no gradient tracking |
| `.to(device)` | Move tensor to CPU/GPU |
| `.item()` | Scalar tensor → Python float |

**Key dtypes:** `float32` (default), `float16` (half precision), `int64`, `bool`


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping — PyTorch not available.')
else:
    # Tensor creation
    t1 = torch.tensor([1.0, 2.0, 3.0])
    t2 = torch.zeros(3, 4)
    t3 = torch.ones(2, 3, dtype=torch.float32)
    t4 = torch.randn(4, 4)  # standard normal
    t5 = torch.arange(12).reshape(3, 4).float()

    print('t1:', t1, t1.dtype)
    print('t2 shape:', t2.shape)
    print('t5:\n', t5)

    # NumPy bridge
    arr = np.array([1, 2, 3])
    t_from_np = torch.from_numpy(arr)   # shares memory!
    np_from_t = t4.numpy()              # back to numpy (CPU only)
    print('\nFrom NumPy:', t_from_np)
    print('To NumPy shape:', np_from_t.shape)

    # GPU transfer
    t_dev = t4.to(DEVICE)
    print(f'\nTensor on {t_dev.device}: shape {t_dev.shape}')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Autograd — how backprop works
    x = torch.tensor(3.0, requires_grad=True)
    y = x**2 + 2*x + 1  # y = (x+1)^2
    y.backward()          # dy/dx = 2x + 2 = 2*3 + 2 = 8
    print(f'x={x.item()}, y={y.item()}, dy/dx={x.grad.item()}')

    # Multivariable gradient
    a = torch.tensor([2.0, 3.0], requires_grad=True)
    b = torch.tensor([4.0, 5.0], requires_grad=True)
    z = (a * b).sum()  # z = a0*b0 + a1*b1
    z.backward()
    print(f'dz/da = {a.grad}')  # should be [4, 5]
    print(f'dz/db = {b.grad}')  # should be [2, 3]

    # torch.no_grad — inference mode
    with torch.no_grad():
        val = (x**2).item()
    print(f'\nIn no_grad context: {val} (no gradient tracked)')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Manual gradient descent — understand autograd before using optimizers
    # Fit y = 2x + 1 from noisy data
    X_true = torch.linspace(-2, 2, 100).reshape(-1, 1)
    y_true = 2 * X_true + 1 + torch.randn_like(X_true) * 0.3

    w = torch.tensor([[0.5]], requires_grad=True)
    b = torch.tensor([0.0],   requires_grad=True)

    lr = 0.01
    losses = []

    for epoch in range(200):
        y_pred = X_true @ w + b         # forward pass
        loss   = ((y_pred - y_true)**2).mean()  # MSE loss

        loss.backward()                  # compute gradients

        with torch.no_grad():            # update weights (not tracked)
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()              # MUST zero gradients each step
            b.grad.zero_()

        losses.append(loss.item())

    print(f'Learned: w={w.item():.4f}, b={b.item():.4f}')
    print(f'True:    w=2.0,        b=1.0')
    print(f'Final loss: {losses[-1]:.4f}')

## Exercises

1. Create a 5x5 tensor of random values, compute its element-wise square, and verify the gradient via autograd.
2. Implement the chain rule manually: if `z = sin(x²)`, compute `dz/dx` at `x=π/4` using autograd and verify analytically.
3. Time a matrix multiply of two 1000x1000 tensors on CPU vs GPU (if available).
4. Implement a linear regression fit using autograd (no nn.Module) with momentum-based gradient descent.

## Summary

- Tensors = NumPy arrays with GPU support and autograd.
- `requires_grad=True` → track operations → `.backward()` → `.grad` contains gradient.
- Always call `.zero_()` on gradients between steps — PyTorch accumulates them by default.
- Use `torch.no_grad()` during inference to save memory and compute.

---


### Exercise and Challenge Solutions — Section 1


In [ ]:
if TORCH_AVAILABLE:
    # Exercise 2: chain rule — dz/dx of sin(x^2)
    x_val = torch.tensor(np.pi / 4, requires_grad=True)
    z = torch.sin(x_val**2)
    z.backward()
    # Analytical: dz/dx = cos(x^2) * 2x
    import math
    x_py = math.pi / 4
    analytic = math.cos(x_py**2) * 2 * x_py
    print(f'Autograd dz/dx: {x_val.grad.item():.6f}')
    print(f'Analytic dz/dx: {analytic:.6f}')

    # Exercise 3: timing
    import time
    A = torch.randn(1000, 1000)
    B = torch.randn(1000, 1000)
    t0 = time.perf_counter()
    for _ in range(10): C = A @ B
    print(f'\nCPU matmul (10x): {(time.perf_counter()-t0)*1000:.1f} ms')
    if torch.cuda.is_available():
        A_gpu = A.cuda(); B_gpu = B.cuda()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(10): C = A_gpu @ B_gpu
        torch.cuda.synchronize()
        print(f'GPU matmul (10x): {(time.perf_counter()-t0)*1000:.1f} ms')

# Section 2 — Building Neural Networks: nn.Module

## Concept

`nn.Module` is the base class for all neural network layers and models.
Override `__init__` (define layers) and `forward` (define computation).

## Technical Deep Dive

**Key layers:**

| Layer | Use Case |
|---|---|
| `nn.Linear(in, out)` | Fully connected layer |
| `nn.Conv2d(in, out, k)` | 2D convolution |
| `nn.ReLU()` / `F.relu` | Activation function |
| `nn.Sigmoid()` / `nn.Tanh()` | Gating / output activations |
| `nn.Dropout(p)` | Regularization |
| `nn.BatchNorm1d/2d` | Normalize activations |
| `nn.Embedding(n, d)` | Integer → dense vector |
| `nn.LSTM` / `nn.GRU` | Sequence models |

**Sequential vs custom Module:**
- `nn.Sequential`: simple linear stack.
- Custom `Module`: skip connections, multiple inputs/outputs, conditional logic.


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # nn.Sequential — simple MLP
    mlp = nn.Sequential(
        nn.Linear(10, 64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid()
    ).to(DEVICE)

    print('Sequential MLP:')
    print(mlp)

    # Count parameters
    total_params = sum(p.numel() for p in mlp.parameters())
    trainable    = sum(p.numel() for p in mlp.parameters() if p.requires_grad)
    print(f'\nTotal params:     {total_params:,}')
    print(f'Trainable params: {trainable:,}')

    # Forward pass with dummy data
    x_dummy = torch.randn(32, 10).to(DEVICE)  # batch of 32, 10 features
    with torch.no_grad():
        out = mlp(x_dummy)
    print(f'\nInput shape: {x_dummy.shape} → Output shape: {out.shape}')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Custom nn.Module — full flexibility
    class MLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3, activation=nn.ReLU):
            super().__init__()
            dims = [input_dim] + hidden_dims
            layers = []
            for i in range(len(dims) - 1):
                layers += [
                    nn.Linear(dims[i], dims[i+1]),
                    nn.BatchNorm1d(dims[i+1]),
                    activation(),
                    nn.Dropout(dropout)
                ]
            self.hidden = nn.Sequential(*layers)
            self.output = nn.Linear(hidden_dims[-1], output_dim)

        def forward(self, x):
            x = self.hidden(x)
            return self.output(x)

    model = MLP(input_dim=10, hidden_dims=[128, 64, 32], output_dim=1, dropout=0.2).to(DEVICE)
    print(model)

    # Weight initialization
    def init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)

    model.apply(init_weights)
    print('\nWeights initialized (Xavier uniform).')

## Exercises

1. Build a neural network for multiclass classification (10 classes, 20 input features).
2. Implement a ResNet-style residual block as a custom `nn.Module` (y = F(x) + x).
3. Compare Xavier, Kaiming, and default initialization on a 5-layer MLP.
4. Build a network that accepts two separate inputs and concatenates them before a final layer.

## Summary

- `nn.Module`: define architecture in `__init__`, computation in `forward`.
- `nn.Sequential`: clean for linear stacks.
- Always call `.to(device)` after creating the model.
- `model.parameters()`: all trainable weights for the optimizer.

---


### Exercise and Challenge Solutions — Section 2


In [ ]:
if TORCH_AVAILABLE:
    # Exercise 2: Residual block
    class ResidualBlock(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.block = nn.Sequential(
                nn.Linear(dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Linear(dim, dim),
                nn.BatchNorm1d(dim)
            )
            self.relu = nn.ReLU()

        def forward(self, x):
            return self.relu(x + self.block(x))  # residual connection

    res = ResidualBlock(64).to(DEVICE)
    x_test = torch.randn(16, 64).to(DEVICE)
    out = res(x_test)
    print(f'ResidualBlock: {x_test.shape} → {out.shape}')

    # Exercise 4: dual-input network
    class DualInputNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.branch_a = nn.Linear(10, 32)
            self.branch_b = nn.Linear(5, 16)
            self.combine  = nn.Sequential(
                nn.ReLU(),
                nn.Linear(48, 1)
            )

        def forward(self, a, b):
            x = torch.cat([self.branch_a(a), self.branch_b(b)], dim=1)
            return self.combine(x)

    dual = DualInputNet().to(DEVICE)
    a = torch.randn(8, 10).to(DEVICE)
    b = torch.randn(8, 5).to(DEVICE)
    print(f'DualInputNet output: {dual(a, b).shape}')

# Section 3 — The Training Loop

## Concept

Every PyTorch training loop has this structure:

```
for epoch in range(epochs):
    model.train()          # enable dropout, batchnorm training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()      # 1. clear old gradients
        pred = model(X_batch)      # 2. forward pass
        loss = criterion(pred, y)  # 3. compute loss
        loss.backward()            # 4. backpropagate
        optimizer.step()           # 5. update weights
    
    model.eval()           # disable dropout, batchnorm uses running stats
    with torch.no_grad():
        for X_val, y_val in val_loader:
            val_pred = model(X_val)  # validation
```

## Technical Deep Dive

**Common loss functions:**

| Task | Loss | PyTorch |
|---|---|---|
| Binary classification | Binary cross-entropy | `nn.BCELoss` / `nn.BCEWithLogitsLoss` |
| Multiclass | Cross-entropy | `nn.CrossEntropyLoss` |
| Regression | MSE | `nn.MSELoss` |
| Regression (robust) | MAE / Huber | `nn.L1Loss` / `nn.HuberLoss` |
| Multi-label | BCE | `nn.BCEWithLogitsLoss` |

**Tip:** Prefer `BCEWithLogitsLoss` over `BCELoss` + Sigmoid — numerically more stable.


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    # Dataset
    X_np, y_np = make_classification(n_samples=2000, n_features=20, n_informative=10,
                                      n_classes=2, random_state=42)
    X_tr_np, X_va_np, y_tr_np, y_va_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

    sc = StandardScaler()
    X_tr_np = sc.fit_transform(X_tr_np)
    X_va_np = sc.transform(X_va_np)

    # Convert to PyTorch tensors
    X_tr_t = torch.FloatTensor(X_tr_np).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_tr_np).unsqueeze(1).to(DEVICE)
    X_va_t = torch.FloatTensor(X_va_np).to(DEVICE)
    y_va_t = torch.FloatTensor(y_va_np).unsqueeze(1).to(DEVICE)

    # DataLoader — automatic batching
    train_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds   = TensorDataset(X_va_t, y_va_t)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)

    print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Full training loop with validation
    model_clf = MLP(input_dim=20, hidden_dims=[64, 32], output_dim=1, dropout=0.3).to(DEVICE)
    criterion  = nn.BCEWithLogitsLoss()
    optimizer  = Adam(model_clf.parameters(), lr=1e-3, weight_decay=1e-4)

    def accuracy(logits, targets):
        return ((logits > 0).float() == targets).float().mean().item()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    N_EPOCHS = 30
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(N_EPOCHS):
        # --- Training ---
        model_clf.train()
        tr_losses, tr_accs = [], []
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            pred = model_clf(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            optimizer.step()
            tr_losses.append(loss.item())
            tr_accs.append(accuracy(pred, y_b))

        # --- Validation ---
        model_clf.eval()
        va_losses, va_accs = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                pred = model_clf(X_b)
                loss = criterion(pred, y_b)
                va_losses.append(loss.item())
                va_accs.append(accuracy(pred, y_b))

        # Record metrics
        tr_loss = np.mean(tr_losses); tr_acc = np.mean(tr_accs)
        va_loss = np.mean(va_losses); va_acc = np.mean(va_accs)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        # Early stopping checkpoint
        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state = {k: v.clone() for k, v in model_clf.state_dict().items()}

        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | '
                  f'Train loss={tr_loss:.4f} acc={tr_acc:.4f} | '
                  f'Val   loss={va_loss:.4f} acc={va_acc:.4f}')

    # Restore best model
    model_clf.load_state_dict(best_state)
    print('\nRestored best model.')

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['val_loss'],   label='Val Loss',   linewidth=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Training History — Loss')
    axes[0].legend()

    axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
    axes[1].plot(history['val_acc'],   label='Val Acc',   linewidth=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training History — Accuracy')
    axes[1].legend()

    plt.tight_layout(); plt.show()

## Exercises

1. Add gradient clipping (`nn.utils.clip_grad_norm_`) to the training loop.
2. Implement early stopping: stop training if validation loss doesn't improve for 5 epochs.
3. Save and load model weights using `torch.save` / `torch.load`.
4. Train a regression MLP on the sklearn diabetes dataset — report test RMSE.

## Summary

- Training loop: zero_grad → forward → loss → backward → step.
- Use `model.train()` / `model.eval()` — affects Dropout and BatchNorm.
- Checkpoint the best model using `model.state_dict()`.
- `BCEWithLogitsLoss` is more stable than `BCELoss` + Sigmoid.

---


### Exercise and Challenge Solutions — Section 3


In [ ]:
if TORCH_AVAILABLE:
    # Exercise 1: gradient clipping
    model_gc = MLP(20, [64, 32], 1, 0.3).to(DEVICE)
    opt_gc = Adam(model_gc.parameters(), lr=1e-3)
    model_gc.train()
    for X_b, y_b in train_loader:
        opt_gc.zero_grad()
        loss = criterion(model_gc(X_b), y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model_gc.parameters(), max_norm=1.0)  # clip
        opt_gc.step()
        break
    print('Gradient clipping applied.')

    # Exercise 3: save/load
    torch.save(model_clf.state_dict(), '/tmp/model_weights.pt')
    model_loaded = MLP(20, [64, 32], 1, 0.3).to(DEVICE)
    model_loaded.load_state_dict(torch.load('/tmp/model_weights.pt', map_location=DEVICE))
    model_loaded.eval()
    with torch.no_grad():
        loaded_acc = accuracy(model_loaded(X_va_t), y_va_t)
    print(f'Loaded model val accuracy: {loaded_acc:.4f}')

    # Exercise 4: regression on diabetes dataset
    from sklearn.datasets import load_diabetes
    diab = load_diabetes()
    X_d, y_d = diab.data, diab.target
    X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(X_d, y_d, test_size=0.2, random_state=42)
    sc_d = StandardScaler()
    X_d_tr = sc_d.fit_transform(X_d_tr); X_d_te = sc_d.transform(X_d_te)

    X_d_tr_t = torch.FloatTensor(X_d_tr).to(DEVICE)
    y_d_tr_t = torch.FloatTensor(y_d_tr).unsqueeze(1).to(DEVICE)
    X_d_te_t = torch.FloatTensor(X_d_te).to(DEVICE)

    reg_net = MLP(10, [64, 32], 1, 0.2).to(DEVICE)
    opt_reg = Adam(reg_net.parameters(), lr=1e-3)
    mse_loss = nn.MSELoss()
    ds_reg = TensorDataset(X_d_tr_t, y_d_tr_t)
    ld_reg = DataLoader(ds_reg, batch_size=32, shuffle=True)

    for _ in range(100):
        reg_net.train()
        for Xb, yb in ld_reg:
            opt_reg.zero_grad(); mse_loss(reg_net(Xb), yb).backward(); opt_reg.step()

    reg_net.eval()
    with torch.no_grad():
        y_pred_reg = reg_net(X_d_te_t).cpu().numpy().ravel()
    rmse = np.sqrt(np.mean((y_pred_reg - y_d_te)**2))
    print(f'Diabetes regression RMSE: {rmse:.2f}')

# Section 4 — Regularization: Dropout, Batch Norm, Weight Decay

## Concept

Regularization prevents overfitting — the gap between training and validation performance.

## Technical Deep Dive

| Technique | How it works | Effect |
|---|---|---|
| **Dropout** | Randomly zeros out p% of neurons per forward pass | Forces redundant representations |
| **Batch Normalization** | Normalizes layer inputs per mini-batch | Faster training, acts as regularizer |
| **Weight Decay (L2)** | Adds `λ||w||²` to loss | Shrinks weights, equivalent to L2 in SGD |
| **Early Stopping** | Stop when val loss plateaus | Prevents memorizing training data |
| **Data Augmentation** | Transform training samples | More effective diversity |
| **Label Smoothing** | Soft targets instead of hard 0/1 | Improves calibration |

**Dropout details:**
- Training: scale surviving activations by `1/(1-p)` to keep expected value.
- Inference: use all neurons (disabled automatically with `model.eval()`).

**Batch Normalization:**
- Normalizes per feature per mini-batch during training.
- Uses running mean/var during inference.
- Place before or after activation (both conventions exist).


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    def train_model(model, optimizer, n_epochs=40, verbose=False):
        criterion_ce = nn.BCEWithLogitsLoss()
        history = {'train': [], 'val': []}
        for epoch in range(n_epochs):
            model.train()
            tr_loss = []
            for Xb, yb in train_loader:
                optimizer.zero_grad()
                loss = criterion_ce(model(Xb), yb)
                loss.backward()
                optimizer.step()
                tr_loss.append(loss.item())
            model.eval()
            va_loss = []
            with torch.no_grad():
                for Xb, yb in val_loader:
                    va_loss.append(criterion_ce(model(Xb), yb).item())
            history['train'].append(np.mean(tr_loss))
            history['val'].append(np.mean(va_loss))
        return history

    # No regularization (overfit)
    m_none = nn.Sequential(
        nn.Linear(20,256), nn.ReLU(), nn.Linear(256,256), nn.ReLU(),
        nn.Linear(256,256), nn.ReLU(), nn.Linear(256,1)
    ).to(DEVICE)
    h_none = train_model(m_none, Adam(m_none.parameters(), lr=1e-3))

    # With dropout + weight decay
    m_reg = nn.Sequential(
        nn.Linear(20,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(256,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(256,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(256,1)
    ).to(DEVICE)
    h_reg = train_model(m_reg, Adam(m_reg.parameters(), lr=1e-3, weight_decay=1e-3))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(h_none['train'], 'b-',   label='No Reg — Train')
    ax.plot(h_none['val'],   'b--',  label='No Reg — Val')
    ax.plot(h_reg['train'],  'r-',   label='BN+Dropout+WD — Train')
    ax.plot(h_reg['val'],    'r--',  label='BN+Dropout+WD — Val')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Regularization Effect')
    ax.legend()
    plt.tight_layout(); plt.show()

## Summary

- Dropout: strong regularizer, disable with `model.eval()`.
- BatchNorm: stabilizes training, often more important than dropout.
- Weight decay: set via `optimizer = Adam(..., weight_decay=1e-4)`.
- Use all three together for deep networks.

---


# Section 5 — Optimizers and Learning Rate Scheduling

## Concept

The optimizer updates weights based on gradients.
The learning rate (LR) is the most important hyperparameter in deep learning.

## Technical Deep Dive

| Optimizer | Key Idea | Best For |
|---|---|---|
| SGD | Gradient descent, optional momentum | Computer vision (with tuning) |
| Adam | Adaptive per-parameter LR + momentum | Default choice |
| AdamW | Adam + decoupled weight decay | Transformers, NLP |
| RMSprop | Adaptive LR, no momentum term | RNNs |
| LARS / LAMB | Large batch distributed training | Very large batches |

**LR Schedules:**

| Schedule | Description |
|---|---|
| `StepLR` | Multiply LR by gamma every N epochs |
| `CosineAnnealingLR` | LR follows cosine curve to min |
| `OneCycleLR` | Warm up → peak → anneal (1cycle policy) |
| `ReduceLROnPlateau` | Reduce LR when metric stagnates |
| `LinearWarmup + Cosine` | Standard for Transformers |


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Visualize LR schedules
    n_epochs = 50
    dummy_model = nn.Linear(10, 1)
    schedulers = {
        'StepLR':          StepLR(Adam(dummy_model.parameters(), lr=0.1), step_size=10, gamma=0.5),
        'CosineAnnealing': CosineAnnealingLR(Adam(dummy_model.parameters(), lr=0.1), T_max=n_epochs)
    }

    fig, ax = plt.subplots(figsize=(9, 4))
    for name, sched in schedulers.items():
        lrs = []
        for _ in range(n_epochs):
            lrs.append(sched.get_last_lr()[0])
            sched.step()
        ax.plot(lrs, label=name, linewidth=2)

    # OneCycle (needs steps_per_epoch)
    spe = len(train_loader)
    oc_opt = Adam(dummy_model.parameters(), lr=0.01)
    oc = OneCycleLR(oc_opt, max_lr=0.1, steps_per_epoch=spe, epochs=n_epochs)
    oc_lrs = []
    for _ in range(n_epochs * spe):
        oc_lrs.append(oc.get_last_lr()[0])
        oc_opt.zero_grad()
        (dummy_model(torch.randn(1, 10))**2).sum().backward()
        oc_opt.step()
        oc.step()
    ax.plot(np.linspace(0, n_epochs, len(oc_lrs)), oc_lrs, label='OneCycleLR', linewidth=2)

    ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
    ax.set_title('LR Schedule Comparison')
    ax.legend()
    plt.tight_layout(); plt.show()

## Summary

- Adam with `lr=1e-3` is a safe default.
- AdamW is preferred for transformers and modern architectures.
- `OneCycleLR` achieves fast convergence with proper warm-up and annealing.
- `ReduceLROnPlateau` is useful when you don't know in advance how many epochs you need.

---


# Section 6 — Convolutional Neural Networks (CNN)

## Concept

CNNs exploit **spatial locality** and **parameter sharing**.
A convolutional layer learns a filter (kernel) that slides over the input, detecting features regardless of position.

## Technical Deep Dive

**Conv2d parameters:**
```
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

Output size: `(W - K + 2P) / S + 1` where W=input, K=kernel, P=padding, S=stride.

**Standard CNN block:**
```
Conv2d → BatchNorm2d → ReLU → (Pooling)
```

**Pooling:**
- `MaxPool2d(2, 2)`: take the maximum in each 2x2 window → halves spatial dimensions.
- `AvgPool2d`: global average pooling before classifier head.
- `AdaptiveAvgPool2d((1,1))`: output always 1x1 regardless of input size.


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    class SimpleCNN(nn.Module):
        '''CNN for 28x28 grayscale images (e.g., MNIST-style).'''
        def __init__(self, n_classes=10):
            super().__init__()
            self.features = nn.Sequential(
                # Block 1: 1x28x28 → 32x14x14
                nn.Conv2d(1, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),

                # Block 2: 32x14x14 → 64x7x7
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),

                # Block 3: 64x7x7 → 128x7x7
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU()
            )
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, n_classes)
            )

        def forward(self, x):
            return self.classifier(self.features(x))

    cnn = SimpleCNN(n_classes=10).to(DEVICE)

    # Trace the shapes
    x_img = torch.randn(8, 1, 28, 28).to(DEVICE)  # batch of 8 grayscale 28x28
    print('Input shape:', x_img.shape)
    for name, layer in [('features', cnn.features), ('classifier', cnn.classifier)]:
        x_img = layer(x_img)
        print(f'After {name}: {x_img.shape}')

    total = sum(p.numel() for p in cnn.parameters())
    print(f'\nTotal params: {total:,}')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Train CNN on synthetic 28x28 image data (random, just to show the pipeline)
    print('Generating synthetic image dataset...')
    n_img = 2000
    X_img = torch.randn(n_img, 1, 28, 28)
    y_img = torch.randint(0, 10, (n_img,))

    img_ds = TensorDataset(X_img, y_img)
    tr_img = DataLoader(img_ds[:1600], batch_size=64, shuffle=True)
    va_img = DataLoader(img_ds[1600:], batch_size=128)

    cnn_model = SimpleCNN(n_classes=10).to(DEVICE)
    opt_cnn   = AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
    ce_loss   = nn.CrossEntropyLoss()
    scheduler = CosineAnnealingLR(opt_cnn, T_max=10)

    for epoch in range(10):
        cnn_model.train()
        for Xb, yb in tr_img:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt_cnn.zero_grad()
            loss = ce_loss(cnn_model(Xb), yb)
            loss.backward()
            opt_cnn.step()
        scheduler.step()

        if (epoch+1) % 5 == 0:
            cnn_model.eval()
            correct = 0; total = 0
            with torch.no_grad():
                for Xb, yb in va_img:
                    Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                    preds = cnn_model(Xb).argmax(dim=1)
                    correct += (preds == yb).sum().item()
                    total += len(yb)
            print(f'Epoch {epoch+1:2d} | Val acc: {correct/total:.4f} (random ~0.10 expected)')

## Summary

- Conv2d learns spatial features via shared filters.
- Conv → BN → ReLU → Pool is the standard building block.
- `AdaptiveAvgPool2d((1,1))` flattens any spatial resolution to a single vector.
- CNNs need far fewer parameters than MLPs for images due to weight sharing.

---


# Section 7 — Transfer Learning

## Concept

Transfer learning reuses a model trained on a large dataset (e.g., ImageNet)
as a starting point for a new, often smaller task.

**Two strategies:**
1. **Feature extraction**: freeze pretrained weights, only train the new head.
2. **Fine-tuning**: unfreeze (some) pretrained layers and train with a small LR.

## Technical Deep Dive

```python
# Load pretrained model
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Strategy 1: freeze all, replace head
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, n_classes)

# Strategy 2: unfreeze last 2 layers + replace head
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc = nn.Linear(model.fc.in_features, n_classes)

# Train with different LRs for frozen vs trainable
optimizer = Adam([
    {'params': model.fc.parameters(),    'lr': 1e-3},
    {'params': model.layer4.parameters(),'lr': 1e-4}
])
```


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    try:
        from torchvision.models import resnet18, ResNet18_Weights
        import torchvision.transforms as T

        # Load pretrained ResNet-18
        model_pt = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        # Strategy 1: Feature extraction — freeze all
        for param in model_pt.parameters():
            param.requires_grad = False

        # Replace classifier head for 5-class task
        n_classes_new = 5
        in_features = model_pt.fc.in_features
        model_pt.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes_new)
        )

        model_pt = model_pt.to(DEVICE)

        trainable = sum(p.numel() for p in model_pt.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model_pt.parameters())
        print(f'ResNet-18 Transfer Learning:')
        print(f'  Total params:     {total:,}')
        print(f'  Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)')

        # Strategy 2: Fine-tune last block + head
        for param in model_pt.layer4.parameters():
            param.requires_grad = True

        trainable2 = sum(p.numel() for p in model_pt.parameters() if p.requires_grad)
        print(f'  After unfreezing layer4: {trainable2:,} trainable ({100*trainable2/total:.1f}%)')

        # Differential learning rates
        optimizer_pt = AdamW([
            {'params': model_pt.fc.parameters(),     'lr': 1e-3},
            {'params': model_pt.layer4.parameters(), 'lr': 1e-4}
        ], weight_decay=1e-4)

        print('\nDifferential LR optimizer configured.')

        # Test forward pass with ImageNet input size
        x_imagenet = torch.randn(4, 3, 224, 224).to(DEVICE)
        with torch.no_grad():
            out = model_pt(x_imagenet)
        print(f'Output shape: {out.shape}  (batch=4, n_classes={n_classes_new})')

    except ImportError:
        print('torchvision not installed. Run: pip install torchvision')
        print('Transfer learning uses pretrained ResNet/EfficientNet/ViT from torchvision.models')

## Best Practices for Transfer Learning

1. **Always start with feature extraction** (frozen backbone) — faster and avoids breaking pretrained features.
2. **Use 10x lower LR** for fine-tuned layers vs the new head.
3. **Augment training data** — pretrained models are more robust to augmentation.
4. **Use the same normalization** as the pretrained model (ImageNet: mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]).
5. **Unfreeze progressively** — start from the top layers and work down.

---


# Section 8 — Recurrent Networks: LSTM and GRU

## Concept

RNNs process **sequential data** by maintaining a hidden state across time steps.

**LSTM** (Long Short-Term Memory) adds **gates** to control information flow:
- Forget gate: what to discard from cell state.
- Input gate: what new information to store.
- Output gate: what to expose from cell state.

**GRU** (Gated Recurrent Unit) is a simplified LSTM with fewer parameters:
two gates (reset, update) instead of three — often matches LSTM performance.


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    class SequenceClassifier(nn.Module):
        def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, n_classes, dropout=0.3, rnn_type='lstm'):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            rnn_cls = nn.LSTM if rnn_type == 'lstm' else nn.GRU
            self.rnn = rnn_cls(
                embed_dim, hidden_dim, num_layers=n_layers,
                batch_first=True, dropout=dropout if n_layers > 1 else 0,
                bidirectional=True
            )
            self.dropout = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden_dim * 2, n_classes)  # *2 for bidirectional

        def forward(self, x):
            emb = self.dropout(self.embedding(x))      # (batch, seq_len, embed_dim)
            out, _ = self.rnn(emb)                      # (batch, seq_len, hidden*2)
            last = out[:, -1, :]                        # take last timestep
            return self.fc(self.dropout(last))

    VOCAB_SIZE = 10000
    EMBED_DIM  = 128
    HIDDEN_DIM = 256
    N_LAYERS   = 2
    N_CLASSES  = 2
    SEQ_LEN    = 100

    lstm_model = SequenceClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, N_CLASSES, rnn_type='lstm').to(DEVICE)
    gru_model  = SequenceClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, N_CLASSES, rnn_type='gru').to(DEVICE)

    # Dummy batch: 32 sequences of length 100
    x_seq = torch.randint(1, VOCAB_SIZE, (32, SEQ_LEN)).to(DEVICE)

    with torch.no_grad():
        lstm_out = lstm_model(x_seq)
        gru_out  = gru_model(x_seq)

    lstm_params = sum(p.numel() for p in lstm_model.parameters())
    gru_params  = sum(p.numel() for p in gru_model.parameters())

    print(f'LSTM output: {lstm_out.shape} | params: {lstm_params:,}')
    print(f'GRU  output: {gru_out.shape}  | params: {gru_params:,}')
    print(f'GRU param reduction: {100*(1-gru_params/lstm_params):.1f}%')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # LSTM for time series forecasting (sine wave prediction)
    class LSTMForecaster(nn.Module):
        def __init__(self, input_size, hidden_size, n_layers, horizon):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, n_layers, batch_first=True, dropout=0.1)
            self.fc   = nn.Linear(hidden_size, horizon)

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :])  # output from last timestep

    # Generate sine wave data
    t = np.linspace(0, 100, 1000)
    y_ts = np.sin(t) + 0.1 * np.random.randn(len(t))

    LOOKBACK = 30; HORIZON = 5
    X_ts = np.array([y_ts[i:i+LOOKBACK] for i in range(len(y_ts)-LOOKBACK-HORIZON)])
    Y_ts = np.array([y_ts[i+LOOKBACK:i+LOOKBACK+HORIZON] for i in range(len(y_ts)-LOOKBACK-HORIZON)])

    X_ts_t = torch.FloatTensor(X_ts).unsqueeze(-1)  # (samples, seq_len, 1)
    Y_ts_t = torch.FloatTensor(Y_ts)                 # (samples, horizon)

    ts_ds = TensorDataset(X_ts_t, Y_ts_t)
    ts_loader = DataLoader(ts_ds[:800], batch_size=32, shuffle=True)

    forecaster = LSTMForecaster(1, 64, 2, HORIZON).to(DEVICE)
    opt_ts = Adam(forecaster.parameters(), lr=1e-3)
    mse = nn.MSELoss()

    for epoch in range(20):
        forecaster.train()
        losses = []
        for Xb, yb in ts_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt_ts.zero_grad()
            loss = mse(forecaster(Xb), yb)
            loss.backward()
            opt_ts.step()
            losses.append(loss.item())

    # Evaluate
    forecaster.eval()
    with torch.no_grad():
        X_test_ts = X_ts_t[800:].to(DEVICE)
        Y_test_ts = Y_ts_t[800:]
        preds_ts = forecaster(X_test_ts).cpu().numpy()

    rmse_ts = np.sqrt(np.mean((preds_ts - Y_test_ts.numpy())**2))
    print(f'Time series forecasting RMSE: {rmse_ts:.4f}')

## Summary

- LSTM and GRU handle sequential data by maintaining state across time steps.
- Use `bidirectional=True` when future context is available (not for forecasting).
- GRU: fewer params, similar performance — good default for smaller datasets.
- For modern NLP: prefer Transformer-based models over RNNs.

---


# Section 9 — Attention and Transformer Basics

## Concept

**Attention** allows the model to focus on relevant parts of the input.
The **Transformer** architecture replaces recurrence with multi-head self-attention
and position-wise feed-forward networks — enabling massive parallelism.

## Technical Deep Dive

**Scaled Dot-Product Attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query): what are we looking for?
- **K** (Key): what does each position offer?
- **V** (Value): what information to extract?

**Transformer block:**
```
x → MultiHeadAttention → AddNorm → FFN → AddNorm → output
```

**Positional encoding** — since attention is permutation-invariant,
we inject position information via sinusoidal encodings or learned embeddings.


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Scaled dot-product attention — from scratch
    def scaled_dot_product_attention(Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, V), weights

    # Demo
    batch, seq_len, d_model = 2, 5, 16
    Q = torch.randn(batch, seq_len, d_model)
    K = torch.randn(batch, seq_len, d_model)
    V = torch.randn(batch, seq_len, d_model)

    output, weights = scaled_dot_product_attention(Q, K, V)
    print(f'Attention output shape: {output.shape}')
    print(f'Attention weights shape: {weights.shape}')
    print(f'Weights sum to 1: {weights[0].sum(dim=-1).round(decimals=4)}')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # PyTorch built-in Transformer encoder for classification
    class TransformerClassifier(nn.Module):
        def __init__(self, vocab_size, d_model, n_heads, n_layers, n_classes, max_seq_len=128, dropout=0.1):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
            self.pos_embed  = nn.Embedding(max_seq_len, d_model)  # learned positional

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads,
                dim_feedforward=d_model*4, dropout=dropout,
                batch_first=True
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
            self.fc = nn.Linear(d_model, n_classes)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x, src_key_padding_mask=None):
            positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
            emb = self.dropout(self.embedding(x) + self.pos_embed(positions))
            out = self.transformer(emb, src_key_padding_mask=src_key_padding_mask)
            cls_token = out[:, 0, :]  # use first token as class representation (CLS)
            return self.fc(cls_token)

    transformer_clf = TransformerClassifier(
        vocab_size=10000, d_model=128, n_heads=4, n_layers=2, n_classes=2
    ).to(DEVICE)

    # Count params
    n_params = sum(p.numel() for p in transformer_clf.parameters())
    print(f'Transformer classifier params: {n_params:,}')

    # Forward pass
    x_seq2 = torch.randint(1, 10000, (16, 50)).to(DEVICE)  # batch=16, seq=50
    with torch.no_grad():
        out2 = transformer_clf(x_seq2)
    print(f'Output shape: {out2.shape}')

## Exercises

1. Implement sinusoidal positional encoding (as in the original Attention paper) instead of learned embeddings.
2. Build a simple GPT-style autoregressive Transformer with causal masking.
3. Visualize the attention weights of the trained TransformerClassifier on a test sequence.
4. Use `transformers` (Hugging Face) library to load a pretrained BERT model and fine-tune it for binary classification.

## Summary

- Attention: Q×K similarity → softmax weights → weighted sum of V.
- Multi-head attention: run attention in parallel with different projections.
- Transformers replaced RNNs for NLP and are expanding to vision (ViT), audio, and tabular data.
- Use pretrained HuggingFace models for NLP rather than training from scratch.

---


# Section 10 — Practical Tips and Production Patterns

## Concept

Moving from notebook to production requires:
reproducibility, performance, debugging tools, and deployment hygiene.


In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    import os

    # Complete reproducibility setup
    def set_seed(seed=42):
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        # Deterministic ops (may slow down GPU training)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    set_seed(42)
    print('Reproducibility seed set.')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Monitor gradient norms during training
    def get_grad_norm(model):
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        return total_norm ** 0.5

    # Check for dead neurons (all outputs zero after activation)
    def count_dead_relu(activations):
        return (activations == 0).float().mean().item()

    # Quick model summary
    def model_summary(model, input_shape):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'Input shape:    {input_shape}')
        print(f'Total params:   {total:,}')
        print(f'Trainable:      {trainable:,}')
        print(f'Non-trainable:  {total-trainable:,}')
        print(f'Memory (float32): {total * 4 / 1e6:.2f} MB')

    model_summary(model_clf, (32, 20))

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Custom Dataset — production pattern
    class TabularDataset(Dataset):
        def __init__(self, X: np.ndarray, y: np.ndarray = None, transform=None):
            self.X = torch.FloatTensor(X)
            self.y = torch.FloatTensor(y).unsqueeze(1) if y is not None else None
            self.transform = transform

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            x = self.X[idx]
            if self.transform:
                x = self.transform(x)
            if self.y is not None:
                return x, self.y[idx]
            return x

    # Augmentation transform for tabular data (Gaussian noise)
    def gaussian_noise(x, std=0.05):
        return x + torch.randn_like(x) * std

    tab_ds = TabularDataset(X_tr_np, y_tr_np, transform=gaussian_noise)
    tab_loader = DataLoader(tab_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=False)

    Xb, yb = next(iter(tab_loader))
    print(f'Batch shape: X={Xb.shape}, y={yb.shape}')

In [ ]:
if not TORCH_AVAILABLE:
    print('Skipping.')
else:
    # Model export patterns

    # 1. Save full checkpoint (resume training)
    checkpoint = {
        'epoch':       29,
        'model_state': model_clf.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'val_loss':    best_val_loss,
        'config':      {'hidden_dims': [64, 32], 'input_dim': 20, 'dropout': 0.3}
    }
    torch.save(checkpoint, '/tmp/full_checkpoint.pt')
    print('Full checkpoint saved.')

    # Load checkpoint
    ckpt = torch.load('/tmp/full_checkpoint.pt', map_location=DEVICE)
    model_resumed = MLP(**{k: v for k, v in ckpt['config'].items()}, output_dim=1).to(DEVICE)
    model_resumed.load_state_dict(ckpt['model_state'])
    print(f'Resumed from epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f}')

    # 2. TorchScript — serialize for C++ deployment
    model_clf.eval()
    try:
        scripted = torch.jit.script(model_clf)
        scripted.save('/tmp/model_scripted.pt')
        print('TorchScript model saved.')
    except Exception as e:
        print(f'TorchScript note: {e}')

    # 3. Efficient inference
    model_clf.eval()
    X_infer = torch.FloatTensor(X_va_np[:10]).to(DEVICE)
    with torch.no_grad():
        logits = model_clf(X_infer)
        probs  = torch.sigmoid(logits)
        preds  = (probs > 0.5).int()
    print(f'\nInference on 10 samples:')
    print(f'Probabilities: {probs.squeeze().cpu().numpy().round(3)}')
    print(f'Predictions:   {preds.squeeze().cpu().numpy()}')

## Exercises

1. Implement `torch.utils.data.WeightedRandomSampler` to oversample the minority class in an imbalanced dataset.
2. Add `torch.amp` (automatic mixed precision with `float16`) to the training loop and measure speedup on GPU.
3. Use `torch.profiler` to identify the bottleneck in a training iteration.
4. Export a model to ONNX format using `torch.onnx.export` and validate the output.

## Mini Challenge

Build a complete reusable `Trainer` class with:
- `fit(model, train_loader, val_loader, epochs)` → returns history dict
- Gradient clipping
- LR scheduling (cosine)
- Early stopping (patience=5)
- Checkpoint saving (best val loss)
- Mixed precision support (if CUDA)

## Best Practices

- Use `torch.no_grad()` during inference — saves memory and compute.
- `model.eval()` before inference — disables dropout and batchnorm training mode.
- Pin memory (`pin_memory=True`) and use multiple workers (`num_workers=4`) in DataLoader for faster data loading.
- Profile before optimizing — don't guess where the bottleneck is.
- Use `mixed precision` (`torch.amp`) for 2-3x GPU speedup with minimal code change.

## Common Mistakes

- Forgetting `optimizer.zero_grad()` — gradients accumulate across batches.
- Not calling `model.eval()` during inference — dropout gives stochastic predictions.
- Moving tensors between CPU/GPU in the training loop — expensive copy.
- Using `loss.item()` inside the loop before `loss.backward()` — breaks gradient graph.
- Not using `pin_memory` and workers in DataLoader — CPU becomes the bottleneck.

## Summary

- Always set seed for reproducibility.
- Checkpoint full training state (model + optimizer + epoch + metric).
- Custom `Dataset` enables augmentation, lazy loading, and complex data sources.
- TorchScript and ONNX for production deployment.
- Mixed precision training: 2-3x speedup with `torch.amp.autocast`.

---


### Exercise and Challenge Solutions — Section 10


In [ ]:
if TORCH_AVAILABLE:
    class Trainer:
        def __init__(self, criterion, optimizer_cls=Adam, lr=1e-3,
                     max_norm=1.0, patience=5, checkpoint_path='/tmp/best.pt',
                     use_amp=False):
            self.criterion = criterion
            self.optimizer_cls = optimizer_cls
            self.lr = lr
            self.max_norm = max_norm
            self.patience = patience
            self.checkpoint_path = checkpoint_path
            self.use_amp = use_amp and torch.cuda.is_available()
            self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None

        def fit(self, model, train_loader, val_loader, epochs):
            optimizer = self.optimizer_cls(model.parameters(), lr=self.lr)
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
            history = {'train_loss':[], 'val_loss':[]}

            best_val_loss = float('inf')
            no_improve = 0

            for epoch in range(epochs):
                # Training
                model.train()
                tr_losses = []
                for Xb, yb in train_loader:
                    Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                    optimizer.zero_grad()
                    if self.use_amp:
                        with torch.cuda.amp.autocast():
                            loss = self.criterion(model(Xb), yb)
                        self.scaler.scale(loss).backward()
                        self.scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), self.max_norm)
                        self.scaler.step(optimizer)
                        self.scaler.update()
                    else:
                        loss = self.criterion(model(Xb), yb)
                        loss.backward()
                        nn.utils.clip_grad_norm_(model.parameters(), self.max_norm)
                        optimizer.step()
                    tr_losses.append(loss.item())

                # Validation
                model.eval()
                va_losses = []
                with torch.no_grad():
                    for Xb, yb in val_loader:
                        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                        va_losses.append(self.criterion(model(Xb), yb).item())

                tr_loss = np.mean(tr_losses); va_loss = np.mean(va_losses)
                history['train_loss'].append(tr_loss)
                history['val_loss'].append(va_loss)
                scheduler.step()

                # Checkpoint + early stopping
                if va_loss < best_val_loss:
                    best_val_loss = va_loss
                    no_improve = 0
                    torch.save(model.state_dict(), self.checkpoint_path)
                else:
                    no_improve += 1
                    if no_improve >= self.patience:
                        print(f'Early stopping at epoch {epoch+1}')
                        break

                if (epoch+1) % 10 == 0:
                    print(f'Epoch {epoch+1:3d} | train={tr_loss:.4f} val={va_loss:.4f} | no_improve={no_improve}')

            # Restore best
            model.load_state_dict(torch.load(self.checkpoint_path, map_location=DEVICE))
            print(f'Best val loss: {best_val_loss:.4f}')
            return history

    # Demo the Trainer
    set_seed(42)
    test_model = MLP(20, [64, 32], 1, 0.3).to(DEVICE)
    trainer = Trainer(criterion=nn.BCEWithLogitsLoss(), lr=1e-3, patience=5)
    hist = trainer.fit(test_model, train_loader, val_loader, epochs=50)

# Course Summary

| Section | Key Skills |
|---|---|
| 1. Tensors & Autograd | Tensor ops, gradient tracking, manual GD |
| 2. nn.Module | Sequential, custom modules, weight init |
| 3. Training Loop | forward→loss→backward→step, DataLoader, history |
| 4. Regularization | Dropout, BatchNorm, weight decay |
| 5. Optimizers & LR | Adam/AdamW, OneCycleLR, CosineAnnealing |
| 6. CNN | Conv2d, MaxPool, AdaptiveAvgPool, image pipeline |
| 7. Transfer Learning | Feature extraction, fine-tuning, differential LR |
| 8. LSTM/GRU | Sequence modeling, time series forecasting |
| 9. Transformer | Attention, encoder, positional embedding |
| 10. Production | Checkpoint, TorchScript, custom Dataset, Trainer class |

## Architecture Selection Guide

| Data Type | Recommended Architecture |
|---|---|
| Tabular | MLP (with BN + Dropout) or GBM |
| Images | CNN (ResNet, EfficientNet, ViT) |
| Text sequences | Transformer (BERT, GPT, T5) |
| Time series | LSTM / GRU or Temporal Conv Net (TCN) |
| Graph data | Graph Neural Network (GNN) |
| Multi-modal | Shared encoder + task-specific heads |

## Next Steps

- **Hugging Face Transformers** — pretrained BERT, GPT, T5, Llama for NLP
- **PyTorch Lightning** — structured training with less boilerplate
- **ONNX / TorchServe / FastAPI** — model serving
- **MLflow / W&B** — experiment tracking
- **Diffusion Models** — generative AI (Stable Diffusion, DALL-E)

---
